# Using Kaggle notebooks as a model server (with llama.cpp)

![](https://cdn-images-1.medium.com/max/880/1*c0pioKlhVSBeX7ya6iPHsQ.jpeg)

> This notebook is a fork of the original Ollama-based version, adapted to use [llama.cpp](https://github.com/ggml-org/llama.cpp) (`llama-server`) as the inference engine instead of Ollama.

## Prerequisites
- [Kaggle account](https://www.kaggle.com/) to run a [Kaggle (code) notebook](https://www.kaggle.com/code).
- Sign up to [ngrok](https://ngrok.com/) and copy your [authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)

## Setting up the server

### At the Kaggle notebook instance
1. Start the notebook session.
  - This works with GPU or CPU, make sure to use a model/quant that fits the chosen hardware.
2. Set up your environment.
  - Get [llama.cpp](https://github.com/ggml-org/llama.cpp) (`llama-server`), either via a prebuilt CUDA binary (fast) or by compiling from source (fallback).
  - Install ngrok.
  - Load the dependencies.
3. Set the ngrok authtoken and server port.
4. Start the `llama.cpp` server (`llama-server`), which exposes an OpenAI-compatible API and can auto-download GGUF models straight from Hugging Face.
5. Set up the `ngrok` credentials.
6. Open the `ngrok` tunnel.
7. Copy the public URL, it will look something like this: `https://ae63-35-230-113-242.ngrok-free.app`.

After following the steps above, you will be able to query the model hosted by Kaggle through your terminal, a notebook, or any other source.

## Prompting the model

### Local terminal
`llama-server` exposes an **OpenAI-compatible** API at `/v1/chat/completions` (plus a native `/completion` endpoint). You can query it with `curl` or any OpenAI-compatible client, pointing `base_url` at the ngrok URL created above.

### Jupyter notebook
The easiest way to query the llama.cpp server is with the official [`openai` Python client](https://github.com/openai/openai-python), since `llama-server` implements the OpenAI Chat Completions API. You can also use the `requests` library to hit the endpoints directly.

#### Adapted from the original Ollama notebook/blog post.

This version swaps Ollama for `llama.cpp`'s `llama-server`, which:
- Can be obtained as a prebuilt CUDA binary (no compile step), with a from-source build as fallback.
- Can pull GGUF models straight from the Hugging Face Hub with the `-hf <user>/<repo>:<quant>` flag (similar to `ollama run <model>`).
- Speaks the OpenAI Chat Completions API natively, so existing OpenAI-client code keeps working.

# GPU information

In [ ]:
!nvidia-smi

# Setup

In [ ]:
# Build dependencies, kept as a fallback in case the prebuilt binary doesn't work
# (only used if the from-source build below is triggered).
!apt-get update -qq
!apt-get install -y -qq build-essential cmake git libcurl4-openssl-dev

In [ ]:
# Get llama.cpp's `llama-server` (and friends) as a prebuilt CUDA binary -> much faster than compiling.
# Targets Ubuntu x86_64 + CUDA 12.8 (compatible with driver >= 570.15, e.g. the 580.x driver on Kaggle T4 instances).
import subprocess, json, os

os.makedirs('llama.cpp/build/bin', exist_ok=True)

PREBUILT_OK = False
try:
    release = json.loads(subprocess.run(
        ['curl', '-s', 'https://api.github.com/repos/ai-dock/llama.cpp-cuda/releases/latest'],
        capture_output=True, text=True
    ).stdout)
    asset_url = next(
        a['browser_download_url'] for a in release['assets']
        if 'cuda-12.8-amd64.tar.gz' in a['name']
    )
    print(f"Downloading prebuilt llama.cpp binary: {asset_url}")
    subprocess.run(['wget', '-q', asset_url, '-O', 'llama-cpp.tar.gz'], check=True)
    # The archive contains a top-level `cuda-12.8/` folder; strip it so binaries land directly in build/bin
    subprocess.run(['tar', '-xzf', 'llama-cpp.tar.gz', '-C', 'llama.cpp/build/bin', '--strip-components=1'], check=True)
    subprocess.run('chmod +x llama.cpp/build/bin/*', shell=True, check=True)
    PREBUILT_OK = True
except Exception as e:
    print(f"Prebuilt binary download/extract failed ({e}); will build from source instead.")

print("PREBUILT_OK =", PREBUILT_OK)

In [ ]:
# Fallback: build from source if the prebuilt binary failed.
# Restricting to the T4's compute capability (7.5) keeps the build time short (~5 min instead of ~20+).
if not PREBUILT_OK:
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp llama.cpp_src
    !cmake -S llama.cpp_src -B llama.cpp/build \
        -DGGML_CUDA=ON -DLLAMA_CURL=ON -DCMAKE_BUILD_TYPE=Release \
        -DCMAKE_CUDA_ARCHITECTURES=75
    !cmake --build llama.cpp/build --config Release -j $(nproc) --target llama-server llama-cli

In [ ]:
# Sanity check: the prebuilt binary needs its bundled .so files on LD_LIBRARY_PATH to run.
LLAMA_BIN_DIR = os.path.abspath('llama.cpp/build/bin')
!LD_LIBRARY_PATH={LLAMA_BIN_DIR} {LLAMA_BIN_DIR}/llama-server --version

In [ ]:
!pip install -qq pyngrok openai

# Dependencies

In [ ]:
import IPython
import subprocess
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient
from typing import Any, Dict, List, Optional
import subprocess, os, time

env = os.environ.copy()

# llama-server needs its bundled shared libraries (e.g. libllama-server-impl.so) on LD_LIBRARY_PATH
env["LD_LIBRARY_PATH"] = LLAMA_BIN_DIR + os.pathsep + env.get("LD_LIBRARY_PATH", "")

# Note: GGUF models downloaded via `-hf` are cached under the default HF cache
# (~/.cache/huggingface), which lives on the root disk -- NOT under /kaggle/working
# (which has a much smaller ~20GB output quota). Don't redirect HF_HOME there.
env["CUDA_VISIBLE_DEVICES"] = "0,1"


# Auxiliary functions

In [ ]:
LLAMA_SERVER_BIN = os.path.join(LLAMA_BIN_DIR, 'llama-server')


def start_llama_server(
    model_id: str,
    port: str,
    n_gpu_layers: int = 999,
    ctx_size: int = 150000,
    extra_args: Optional[List[str]] = None,
    env: dict = None,
) -> None:
    """Starts the llama.cpp server (`llama-server`).

    Args:
        model_id: A Hugging Face model reference in `<user>/<repo>:<quant>` form
            (passed to `-hf`). llama-server will download the GGUF on first run
            and cache it under the default Hugging Face cache directory.
        port: The port to serve the OpenAI-compatible API on.
        n_gpu_layers: Number of model layers to offload to the GPU (use a large
            number such as 999 to offload everything that fits).
        ctx_size: Context window size.
        extra_args: Any additional CLI flags to pass to `llama-server`.
        env: Environment variables for the subprocess (must include LD_LIBRARY_PATH,
            see the Dependencies cell above).
    """
    cmd = [
        LLAMA_SERVER_BIN,
        '-hf', model_id,
        '--host', '0.0.0.0',
        '--port', port,
        '-ngl', str(n_gpu_layers),
        '-c', str(ctx_size),
        '--flash-attn', 'on',
        # '--split-mode', 'row',
        '--spec-type', 'draft-mtp',
        '--spec-draft-n-max', '3', 
        '--spec-draft-n-min', '1',
        '--split-mode', 'tensor',        # PARALLEL: 2-3x faster prefill
        '--tensor-split', '0.5,0.5',     # Equal split across GPU 0 & 1
        # '--main-gpu', '0'
        '--batch-size', '4096',          # Larger batch for better throughput
        # '--threads', '8',                # CPU threads (Kaggle: 2 vCPUs per GPU)
        # '--threads-batch', '8',          # CPU threads for batch processing
        '--no-mmap',           
        # '--slots', '2',
    ]
    if extra_args:
        cmd.extend(extra_args)

    subprocess.Popen(cmd, env=env)
    print("llama.cpp server starting (downloading/loading the model can take a while on first run)...")


def check_llama_server_port(port: str, timeout: int = 600, interval: int = 5) -> None:
    """Polls until the llama.cpp server is listening on the specified port, or times out."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            result = subprocess.run(['sudo', 'lsof', '-i', '-P', '-n'], capture_output=True, text=True, check=True)
            if any(f":{port} (LISTEN)" in line for line in result.stdout.splitlines()):
                print(f"llama.cpp server is listening on port {port}")
                return
        except subprocess.CalledProcessError as e:
            print(f"Error checking port: {e}")
        time.sleep(interval)
    print(f"Timed out waiting for llama.cpp server to listen on port {port}.")


def setup_ngrok_tunnel(port: str, secret_manager: UserSecretsClient) -> ngrok.NgrokTunnel:
    """Sets up an ngrok tunnel.

    Args:
        port: The port to tunnel.

    Returns:
        The ngrok tunnel object.

    Raises:
        RuntimeError: If the ngrok authtoken is not set.
    """
    ngrok_auth_token = secret_manager.get_secret("NGROK_TOKEN")
    if not ngrok_auth_token:
        raise RuntimeError("NGROK_AUTHTOKEN is not set.")

    ngrok.set_auth_token(ngrok_auth_token)
    tunnel = ngrok.connect(port, host_header=f'localhost:{port}')
    print(f"ngrok tunnel created: {tunnel.public_url}")
    return tunnel

# Parameters

In [ ]:
# llama-server's default port
NGROK_PORT = '8080'

# Hugging Face GGUF reference in <user>/<repo>:<quant> form, same model/quant as the Ollama version
MODEL_ID = 'unsloth/Qwen3.6-35B-A3B-MTP-GGUF:UD-Q4_K_M'

# Adjust based on available VRAM. 999 offloads as many layers as fit on the GPU(s).
N_GPU_LAYERS = 999
CTX_SIZE = 150000

PROMPT = 'Why is the sky blue?'

# Check disk space

`-hf` downloads the GGUF model into the Hugging Face cache on the **root disk** (`/`). Check there's enough free space for the chosen quant before starting the server.

In [ ]:
!df -h / /kaggle/working

# Start the `llama.cpp` server

In [ ]:
start_llama_server(
    model_id=MODEL_ID,
    port=NGROK_PORT,
    n_gpu_layers=N_GPU_LAYERS,
    ctx_size=CTX_SIZE,
    env=env,
)

check_llama_server_port(NGROK_PORT)

# Setup `ngrok` tunnel

In [ ]:
user_secrets = UserSecretsClient()

ngrok_tunnel = setup_ngrok_tunnel(NGROK_PORT, user_secrets)

# Query the server

## From a terminal

`llama-server` exposes an OpenAI-compatible API, so you can query it with `curl` once you have the ngrok URL.

Copy the command below and run it on your terminal (replace `OLLAMA_HOST_URL` with the ngrok URL printed above):

In [ ]:
print(f'''curl {ngrok_tunnel.public_url}/v1/chat/completions \\
  -H "Content-Type: application/json" \\
  -d '{{"model": "{MODEL_ID}", "messages": [{{"role": "user", "content": "{PROMPT}"}}]}}'\n''')

## From a notebook

### Using the OpenAI-compatible client

In [ ]:
from openai import OpenAI

# llama-server speaks the OpenAI Chat Completions API, so the regular OpenAI client works as-is.
# The api_key value is arbitrary; llama-server does not validate it by default.
client = OpenAI(
    base_url=f"{ngrok_tunnel.public_url}/v1",
    api_key="llama-cpp",
)

In [ ]:
def query_llama_server(client: OpenAI, prompt: str, model_id: str, system_prompt: Optional[str] = None) -> None:
    """Queries the llama.cpp server using the OpenAI-compatible client.

    Args:
        client: An OpenAI client pointed at the llama.cpp server.
        prompt: The prompt to send to the model.
        model_id: The model identifier (llama-server ignores this beyond logging,
            since only one model is loaded, but the field is required by the API).
        system_prompt: Optional system prompt.
    """
    try:
        messages = []
        if system_prompt:
            messages.append({'role': 'system', 'content': system_prompt})
        messages.append({'role': 'user', 'content': prompt})

        response = client.chat.completions.create(
            model=model_id,
            messages=messages,
        )
        print("Response from llama.cpp server:")
        print(response.choices[0].message.content)
    except Exception as e:
        print(f"Error querying llama.cpp server: {e}")

In [ ]:
import requests

def wait_for_model_ready(base_url: str, timeout: int = 1800, interval: int = 10) -> None:
    """Polls llama-server's /health endpoint until the model finishes loading."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            r = requests.get(f"{base_url}/health", timeout=5)
            if r.status_code == 200:
                print("Model is ready.")
                return
            print(f"Status: {r.status_code} - {r.json().get('error', {}).get('message', 'loading...')}")
        except requests.exceptions.RequestException as e:
            print(f"Server not reachable yet: {e}")
        time.sleep(interval)
    print("Timed out waiting for model to be ready.")

#### Text prompt

In [ ]:
wait_for_model_ready(f"http://localhost:{NGROK_PORT}")

In [ ]:
query_llama_server(
    client,
    PROMPT,
    MODEL_ID,
)

#### Multilingual prompt

In [ ]:
query_llama_server(
    client,
    "seberapa baik kamu dalam koding?",
    MODEL_ID,
    system_prompt="Kamu adalah asisten yang helpful.",
)